# Tariff

A **tariff** is a versioned list of cost-formula groups. Each version is active from its `start` timestamp until the next one begins.

Within a version, formulas are organised by **cost group**:

| Key | `CostGroup` | Description |
|---|---|---|
| `consumption` | `CONSUMPTION` | Cost per MWh of energy consumed |
| `injection` | `INJECTION` | Revenue/cost per MWh fed back into the grid |
| `capacity` | `CAPACITY` | Cost per MW of peak demand |
| `fixed` | `FIXED` | Recurring charge (daily, monthly, …) |

Each cost group holds one or more **named formulas** (see [formula.ipynb](./formula.ipynb)). The cost group names are just a convention; they have no special meaning to the code, this means you can use consumption based formulas under `fixed` if you want.
The only exception to this rule is `injection`, which will provide the `injection` meter data to the formulas, instead of `consumption`.

The tariff file is a YAML list; each entry needs a `start` timestamp and one or more cost-group keys.

Just like a formula, a tariff has 2 main methods:
- `apply(consumption, injection, start, end, output_resolution)` applies the tariff to the given consumption and injection data, returning a DataFrame with a MultiIndex of cost groups and formula names, with the resulting cost data in € for each timestamp.
- `get_values(start, end, output_resolution, cost_group)` if possible for the given formulas, returns the price per consumption/injection in €/MWh for the given period, as a DataFrame with a MultiIndex of formula names for the given cost group.

## Setup

Register the `spot` price index and create a shared date range with sample meter readings for use throughout the notebook.

In [1]:
import datetime as dt
from zoneinfo import ZoneInfo

import pandas as pd
from helpers import display_yaml  # ty: ignore[unresolved-import]
from isodate import Duration

from energy_cost import CapacityRule, CostGroup, Meter, Tariff, TimeseriesFrame
from energy_cost.index import CSVIndex, Index

# Spot price index — monthly CSV covering 2025-06 to 2026-03
Index.register("spot", CSVIndex("../examples/indexes/foo.csv"))

# Shared query window: December 2025
CET = ZoneInfo("CET")
start = dt.datetime(2025, 12, 1, tzinfo=CET)
end   = dt.datetime(2026,  2, 1, tzinfo=CET)
hourly   = dt.timedelta(hours=1)
weekly   = dt.timedelta(weeks=1)
monthly = Duration(months=1)



# Sample meter readings (15-min, 1 kW load and 0.4 kW solar)
timestamps = pd.date_range(start, end, freq="15min", inclusive="left")
consumption = Meter(power=TimeseriesFrame({"timestamp": timestamps, "value": 0.00025}))
injection   = Meter(power=TimeseriesFrame({"timestamp": timestamps, "value": 0.00010}))

# Helper to convert a power series (MWh) to capacity (MW) in the expected resolution
# A contract does this for you based on the regional capacity rule
capacity_rule = CapacityRule(
    measurement_period=dt.timedelta(minutes=15),
    billing_period=Duration(months=1)
)
consumption = capacity_rule.apply(consumption)

## 1. Simple flat-rate tariff

The minimal tariff has a single version with one `consumption` formula.

In [2]:
display_yaml("../examples/tariffs/simple.yml")

```yaml
- start: 2025-01-01T00:00:00+01:00
  consumption:
    constant_cost: 120.0

```

In [3]:
tariff = Tariff.from_yaml("../examples/tariffs/simple.yml")

# get_values returns the price series (€/MWh) — independent of meter data
tariff.get_values(start, end, output_resolution=hourly, timezone=CET)

,timestamp,total
0,2025-12-01 00:00:00+01:00,120.0
1,2025-12-01 01:00:00+01:00,120.0
2,2025-12-01 02:00:00+01:00,120.0
3,2025-12-01 03:00:00+01:00,120.0
4,2025-12-01 04:00:00+01:00,120.0
...,...,...
1483,2026-01-31 19:00:00+01:00,120.0
1484,2026-01-31 20:00:00+01:00,120.0
1485,2026-01-31 21:00:00+01:00,120.0
1486,2026-01-31 22:00:00+01:00,120.0


In [4]:
# apply returns the actual cost series (€) — based on the meter data and the tariff formulas
tariff.apply(consumption, injection, start, end, output_resolution=weekly, timezone=CET)

,timestamp,consumption,total
,,total,total
0,2025-12-01 00:00:00+01:00,20.16,20.16
1,2025-12-08 00:00:00+01:00,20.16,20.16
2,2025-12-15 00:00:00+01:00,20.16,20.16
3,2025-12-22 00:00:00+01:00,20.16,20.16
4,2025-12-29 00:00:00+01:00,20.16,20.16
5,2026-01-05 00:00:00+01:00,20.16,20.16
6,2026-01-12 00:00:00+01:00,20.16,20.16
7,2026-01-19 00:00:00+01:00,20.16,20.16
8,2026-01-26 00:00:00+01:00,17.28,17.28


## 2. Versioned tariffs

A tariff can have multiple versions, each starting from a `start` date. When you call `get_values` or `apply` with a range that spans version boundaries, the correct version is used for each segment automatically.

In [5]:
display_yaml("../examples/tariffs/dynamic.yml")

```yaml
- start: 2025-01-01T00:00:00+01:00
  consumption:
    constant_cost: 10.0
    variable_costs:
      - index: spot
        scalar: 1.05
- start: 2026-01-01T00:00:00+01:00
  consumption:
    constant_cost: 12.0
    variable_costs:
      - index: spot
        scalar: 1.10
```

Load the tariff and query December 2025 (version 1) and January 2026 (version 2):

In [6]:
tariff = Tariff.from_yaml("../examples/tariffs/dynamic.yml")

# December 2025 — version 1 applies (10 + spot * 1.05)
# january 2026 — version 2 applies (12 + spot * 1.10)
tariff.get_values(start, end, output_resolution=hourly, timezone=CET)

,timestamp,total
0,2025-12-01 00:00:00+01:00,96.1
1,2025-12-01 01:00:00+01:00,115.0
2,2025-12-01 02:00:00+01:00,115.0
3,2025-12-01 03:00:00+01:00,115.0
4,2025-12-01 04:00:00+01:00,115.0
...,...,...
1483,2026-01-31 19:00:00+01:00,144.0
1484,2026-01-31 20:00:00+01:00,144.0
1485,2026-01-31 21:00:00+01:00,144.0
1486,2026-01-31 22:00:00+01:00,144.0


In [7]:
tariff.apply(consumption, injection, start, end, output_resolution=monthly, timezone=CET)

,timestamp,consumption,total
,,total,total
0,2025-12-01 00:00:00+01:00,85.5411,85.5411
1,2026-01-01 00:00:00+01:00,107.1140,107.1140


## 3. Multiple cost groups

A single tariff version can define formulas for several cost groups at once. Each group is queried by passing `cost_group` to `get_values`.

In [8]:
display_yaml("../examples/tariffs/cost_groups.yml")

```yaml
- start: 2025-01-01T00:00:00+01:00
  consumption:
    constant_cost: 10.0
    variable_costs:
      - index: spot
        scalar: 1.05
  injection:
    variable_costs:
      - index: spot
        scalar: -0.95
  fixed:
    period: P1M
    constant_cost: 5.0
  capacity:
    capacity_based: true
    constant_cost: 2.0
```

In [9]:
tariff = Tariff.from_yaml("../examples/tariffs/cost_groups.yml")

# Consumption price (default cost group)
tariff.get_values(start, end, output_resolution=monthly, timezone=CET)

,timestamp,total
0,2025-12-01 00:00:00+01:00,96.1
1,2026-01-01 00:00:00+01:00,115.0


Injection price — the same tariff, different cost group:

In [10]:
tariff.get_values(start, end, output_resolution=monthly, cost_group=CostGroup.INJECTION)

,timestamp,total
0,2025-12-01 23:00:00+00:00,-95.0
1,2026-01-01 23:00:00+00:00,-114.0


In [11]:
# applying this tariff returns both injection and consumption costs and their total
tariff.apply(consumption, injection, start, end, output_resolution=monthly, timezone=CET)

,timestamp,capacity,consumption,fixed,injection,total
,,total,total,total,total,total
0,2025-12-01 00:00:00+01:00,0.002,85.5411,5.0,-28.26516,62.27794
1,2026-01-01 00:00:00+01:00,0.002,101.1630,5.0,-33.91880,72.24620


## 4. Named cost types

A cost group can hold multiple **named** formulas. `get_values` then returns one column per name, plus a `total` column that sums them.

In [12]:
display_yaml("../examples/tariffs/cost_types.yml")

```yaml
# Multiple named cost types within the consumption group
- start: 2025-01-01T00:00:00+01:00
  consumption:
    energy:
      constant_cost: 10.0
      variable_costs:
        - index: spot
          scalar: 1.05
    renewable_certificates:
      constant_cost: 5.0
    grid_fees:
      constant_cost: 8.0
```

In [13]:
tariff = Tariff.from_yaml("../examples/tariffs/cost_types.yml")

# Returns one column per named formula + total
tariff.get_values(start, end, output_resolution=monthly, timezone=CET)

,timestamp,energy,renewable_certificates,grid_fees,total
0,2025-12-01 00:00:00+01:00,96.1,5.0,8.0,109.1
1,2026-01-01 00:00:00+01:00,115.0,5.0,8.0,128.0


In [14]:
tariff.apply(consumption, injection, start, end, output_resolution=monthly, timezone=CET)

timestamp consumption                                   \
                                 energy grid_fees renewable_certificates   
0 2025-12-01 00:00:00+01:00     85.5411     5.952                   3.72   
1 2026-01-01 00:00:00+01:00    101.1630     5.952                   3.72   

                total  
      total     total  
0   95.2131   95.2131  
1  110.8350  110.8350

## 5. More complex formulas

In the above example tariffs, the formulas are simple linear functions of the spot price. But they can be arbitrarily complex — see [formula.ipynb](./formula.ipynb) for details.